In [ ]:
"""
CPH 200C Problem Set 1: Diabetes Readmission Analysis
Megan Gadiparthy - April 24, 2024
"""

# ============================================================================
# SETUP: Install packages and upload data
# ============================================================================

!pip install -q scikit-learn pandas numpy matplotlib seaborn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
from scipy import stats
import zipfile
import warnings
warnings.filterwarnings('ignore')

# Set styling
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

print("=" * 80)
print("CPH 200C - DIABETES READMISSION RISK STRATIFICATION")
print("=" * 80)

# ============================================================================
# Upload the diabetes dataset zip file
# ============================================================================

from google.colab import files
print("\n📤 Please upload: diabetes_130-us_hospitals_for_years_1999-2008.zip")
uploaded = files.upload()

# Extract the zip file
zip_filename = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('.')

print("✓ Dataset extracted!")

# ============================================================================
# LOAD DATA
# ============================================================================

print("\n" + "=" * 80)
print("[STEP 1] Loading Dataset")
print("=" * 80)

df = pd.read_csv('diabetic_data.csv')

print(f"\n✓ Loaded {len(df):,} hospital encounters")
print(f"✓ Unique patients: {df['patient_nbr'].nunique():,}")
print(f"✓ Features: {len(df.columns)}")
print(f"\nTarget variable distribution:")
print(df['readmitted'].value_counts())

# Create binary target: 1 if readmitted <30 days, 0 otherwise
df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)
overall_rate = df['readmitted_30'].mean()

print(f"\n✓ Overall 30-day readmission rate: {overall_rate:.2%}")

# ============================================================================
# 1.1 DATA EXPLORATION
# ============================================================================

print("\n" + "=" * 80)
print("[STEP 2] 1.1 DATA EXPLORATION - Demographics Analysis")
print("=" * 80)

# AGE ANALYSIS
print("\n📊 Readmission Rates by Age Group:")
age_stats = df.groupby('age')['readmitted_30'].agg(['mean', 'count', 'std'])
age_stats.columns = ['rate', 'n', 'std']
age_stats['rate_%'] = age_stats['rate'] * 100
print("\n" + age_stats[['rate_%', 'n']].to_string())

# GENDER ANALYSIS
print("\n📊 Readmission Rates by Gender:")
gender_stats = df[df['gender'].isin(['Male', 'Female'])].groupby('gender')['readmitted_30'].agg(['mean', 'count'])
gender_stats.columns = ['rate', 'n']
gender_stats['rate_%'] = gender_stats['rate'] * 100
print("\n" + gender_stats[['rate_%', 'n']].to_string())

# RACE ANALYSIS
print("\n📊 Readmission Rates by Race:")
race_stats = df[df['race'] != '?'].groupby('race')['readmitted_30'].agg(['mean', 'count'])
race_stats.columns = ['rate', 'n']
race_stats['rate_%'] = race_stats['rate'] * 100
race_stats_sorted = race_stats.sort_values('rate_%', ascending=False)
print("\n" + race_stats_sorted[['rate_%', 'n']].to_string())

# KEY FINDINGS
print("\n" + "=" * 40)
print("KEY FINDINGS:")
print("=" * 40)
highest_age = age_stats['rate_%'].idxmax()
print(f"• Highest age group readmission: {highest_age} ({age_stats.loc[highest_age, 'rate_%']:.1f}%)")

if gender_stats.loc['Female', 'rate_%'] > gender_stats.loc['Male', 'rate_%']:
    print(f"• Females have higher readmission rate than males " +
          f"({gender_stats.loc['Female', 'rate_%']:.1f}% vs {gender_stats.loc['Male', 'rate_%']:.1f}%)")
else:
    print(f"• Males have higher readmission rate than females " +
          f"({gender_stats.loc['Male', 'rate_%']:.1f}% vs {gender_stats.loc['Female', 'rate_%']:.1f}%)")

highest_race = race_stats_sorted.index[0]
print(f"• Highest race group readmission: {highest_race} ({race_stats_sorted.iloc[0]['rate_%']:.1f}%)")

# VISUALIZATION
print("\n📊 Creating demographic visualization...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age plot
age_data = df.groupby('age')['readmitted_30'].mean() * 100
age_data.plot(kind='bar', ax=axes[0], color='#2E86AB', edgecolor='black', linewidth=0.5)
axes[0].set_title('30-Day Readmission Rate by Age Group', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Age Group', fontsize=12)
axes[0].set_ylabel('Readmission Rate (%)', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)
axes[0].axhline(overall_rate * 100, color='red', linestyle='--', linewidth=2, label='Overall Rate', alpha=0.7)
axes[0].legend(fontsize=10)

# Gender plot
gender_data = df[df['gender'].isin(['Male', 'Female'])].groupby('gender')['readmitted_30'].mean() * 100
colors = ['#A23B72', '#5C9EAD']
gender_data.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_title('30-Day Readmission Rate by Gender', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Gender', fontsize=12)
axes[1].set_ylabel('Readmission Rate (%)', fontsize=12)
axes[1].tick_params(axis='x', rotation=0)
axes[1].grid(axis='y', alpha=0.3)
axes[1].axhline(overall_rate * 100, color='red', linestyle='--', linewidth=2, label='Overall Rate', alpha=0.7)
axes[1].legend(fontsize=10)

# Race plot
race_data = df[df['race'] != '?'].groupby('race')['readmitted_30'].mean() * 100
race_data_sorted = race_data.sort_values(ascending=False)
race_data_sorted.plot(kind='bar', ax=axes[2], color='#F18F01', edgecolor='black', linewidth=0.5)
axes[2].set_title('30-Day Readmission Rate by Race', fontsize=14, fontweight='bold', pad=15)
axes[2].set_xlabel('Race', fontsize=12)
axes[2].set_ylabel('Readmission Rate (%)', fontsize=12)
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(axis='y', alpha=0.3)
axes[2].axhline(overall_rate * 100, color='red', linestyle='--', linewidth=2, label='Overall Rate', alpha=0.7)
axes[2].legend(fontsize=10)

plt.tight_layout()
plt.savefig('fig1_demographics.png', dpi=300, bbox_inches='tight')
print("✓ Saved: fig1_demographics.png")
plt.show()

# ============================================================================
# 1.2 DATA PREPROCESSING
# ============================================================================

print("\n" + "=" * 80)
print("[STEP 3] 1.2 Data Preprocessing")
print("=" * 80)

# Select features
drop_cols = ['encounter_id', 'patient_nbr', 'readmitted', 'readmitted_30']
feature_cols = [col for col in df.columns if col not in drop_cols]

print(f"\n📋 Feature Engineering:")
print(f"  Total features: {len(feature_cols)}")

# Identify column types
categorical_cols = df[feature_cols].select_dtypes(include=['object']).columns.tolist()
numerical_cols = df[feature_cols].select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"  Categorical features: {len(categorical_cols)}")
print(f"  Numerical features: {len(numerical_cols)}")

# Handle missing values (marked as '?')
df_clean = df.copy()
missing_summary = []
for col in categorical_cols:
    missing_count = (df_clean[col] == '?').sum()
    if missing_count > 0:
        missing_pct = missing_count / len(df_clean) * 100
        missing_summary.append(f"  {col}: {missing_pct:.1f}% missing")
    df_clean[col] = df_clean[col].replace('?', np.nan)

if missing_summary:
    print(f"\n⚠️  Missing data:")
    for summary in missing_summary[:5]:  # Show top 5
        print(summary)

# Prepare feature matrix
X = df_clean[feature_cols].copy()
y = df_clean['readmitted_30'].copy()

# Encode categorical variables
print(f"\n🔄 Encoding categorical variables...")
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    mask = X[col].notna()
    if mask.sum() > 0:
        X.loc[mask, col] = le.fit_transform(X.loc[mask, col].astype(str))
        label_encoders[col] = le
    X[col] = pd.to_numeric(X[col], errors='coerce')

# Impute missing values
print(f"🔄 Imputing missing values...")
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns,
    index=X.index
)

# Train-validation-test split
print(f"\n📊 Splitting data (60% train, 20% val, 20% test)...")
X_temp, X_test, y_temp, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"  Train: {len(X_train):,} samples ({y_train.mean():.1%} positive)")
print(f"  Val:   {len(X_val):,} samples ({y_val.mean():.1%} positive)")
print(f"  Test:  {len(X_test):,} samples ({y_test.mean():.1%} positive)")

# Scale features for models that need it
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("✓ Preprocessing complete!")

# ============================================================================
# 1.2 MODEL DEVELOPMENT
# ============================================================================

print("\n" + "=" * 80)
print("[STEP 4] 1.2 Model Training")
print("=" * 80)

results = {}

# ---------------------
# Model 1: Logistic Regression
# ---------------------
print("\n🤖 [1/3] Training Logistic Regression...")
print("  Searching hyperparameters: C ∈ {0.001, 0.01, 0.1, 1, 10, 100}")

lr_params = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'max_iter': [1000]
}

lr_grid = GridSearchCV(
    LogisticRegression(random_state=42, solver='lbfgs'),
    lr_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)
lr_grid.fit(X_train_scaled, y_train)

# Evaluate
lr_val_pred = lr_grid.predict_proba(X_val_scaled)[:, 1]
lr_test_pred = lr_grid.predict_proba(X_test_scaled)[:, 1]

lr_val_auc = roc_auc_score(y_val, lr_val_pred)
lr_test_auc = roc_auc_score(y_test, lr_test_pred)

# Bootstrap confidence interval
n_bootstrap = 1000
lr_auc_boot = []
for i in range(n_bootstrap):
    indices = np.random.choice(len(y_test), len(y_test), replace=True)
    lr_auc_boot.append(roc_auc_score(y_test.iloc[indices], lr_test_pred[indices]))
lr_ci = np.percentile(lr_auc_boot, [2.5, 97.5])

results['Logistic Regression'] = {
    'model': lr_grid.best_estimator_,
    'params': lr_grid.best_params_,
    'val_auc': lr_val_auc,
    'test_auc': lr_test_auc,
    'test_ci': lr_ci,
    'test_pred': lr_test_pred
}

print(f"  ✓ Best params: {lr_grid.best_params_}")
print(f"  ✓ Validation AUC: {lr_val_auc:.4f}")
print(f"  ✓ Test AUC: {lr_test_auc:.4f} (95% CI: [{lr_ci[0]:.4f}, {lr_ci[1]:.4f}])")

# ---------------------
# Model 2: Random Forest
# ---------------------
print("\n🌲 [2/3] Training Random Forest...")
print("  Searching hyperparameters (this takes ~2 minutes)...")

rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, 30],
    'min_samples_split': [5, 10]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)
rf_grid.fit(X_train, y_train)

# Evaluate
rf_val_pred = rf_grid.predict_proba(X_val)[:, 1]
rf_test_pred = rf_grid.predict_proba(X_test)[:, 1]

rf_val_auc = roc_auc_score(y_val, rf_val_pred)
rf_test_auc = roc_auc_score(y_test, rf_test_pred)

# Bootstrap CI
rf_auc_boot = []
for i in range(n_bootstrap):
    indices = np.random.choice(len(y_test), len(y_test), replace=True)
    rf_auc_boot.append(roc_auc_score(y_test.iloc[indices], rf_test_pred[indices]))
rf_ci = np.percentile(rf_auc_boot, [2.5, 97.5])

results['Random Forest'] = {
    'model': rf_grid.best_estimator_,
    'params': rf_grid.best_params_,
    'val_auc': rf_val_auc,
    'test_auc': rf_test_auc,
    'test_ci': rf_ci,
    'test_pred': rf_test_pred
}

print(f"  ✓ Best params: {rf_grid.best_params_}")
print(f"  ✓ Validation AUC: {rf_val_auc:.4f}")
print(f"  ✓ Test AUC: {rf_test_auc:.4f} (95% CI: [{rf_ci[0]:.4f}, {rf_ci[1]:.4f}])")

# ---------------------
# Model 3: Neural Network
# ---------------------
print("\n🧠 [3/3] Training Neural Network...")
print("  Searching hyperparameters...")

nn_params = {
    'hidden_layer_sizes': [(50,), (100,), (100, 50)],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

nn_grid = GridSearchCV(
    MLPClassifier(random_state=42, early_stopping=True, max_iter=500),
    nn_params,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)
nn_grid.fit(X_train_scaled, y_train)

# Evaluate
nn_val_pred = nn_grid.predict_proba(X_val_scaled)[:, 1]
nn_test_pred = nn_grid.predict_proba(X_test_scaled)[:, 1]

nn_val_auc = roc_auc_score(y_val, nn_val_pred)
nn_test_auc = roc_auc_score(y_test, nn_test_pred)

# Bootstrap CI
nn_auc_boot = []
for i in range(n_bootstrap):
    indices = np.random.choice(len(y_test), len(y_test), replace=True)
    nn_auc_boot.append(roc_auc_score(y_test.iloc[indices], nn_test_pred[indices]))
nn_ci = np.percentile(nn_auc_boot, [2.5, 97.5])

results['Neural Network'] = {
    'model': nn_grid.best_estimator_,
    'params': nn_grid.best_params_,
    'val_auc': nn_val_auc,
    'test_auc': nn_test_auc,
    'test_ci': nn_ci,
    'test_pred': nn_test_pred
}

print(f"  ✓ Best params: {nn_grid.best_params_}")
print(f"  ✓ Validation AUC: {nn_val_auc:.4f}")
print(f"  ✓ Test AUC: {nn_test_auc:.4f} (95% CI: [{nn_ci[0]:.4f}, {nn_ci[1]:.4f}])")

# Model comparison visualization
print("\n📊 Creating model comparison plot...")

fig, ax = plt.subplots(figsize=(10, 6))

model_names = list(results.keys())
val_aucs = [results[m]['val_auc'] for m in model_names]
test_aucs = [results[m]['test_auc'] for m in model_names]
test_cis = [results[m]['test_ci'] for m in model_names]

x = np.arange(len(model_names))
width = 0.35

bars1 = ax.bar(x - width/2, val_aucs, width, label='Validation AUC',
               color='#2E86AB', alpha=0.8, edgecolor='black', linewidth=1)
bars2 = ax.bar(x + width/2, test_aucs, width, label='Test AUC',
               color='#F18F01', alpha=0.8, edgecolor='black', linewidth=1)

# Error bars on test AUC
errors = [[test_aucs[i] - test_cis[i][0] for i in range(len(test_aucs))],
          [test_cis[i][1] - test_aucs[i] for i in range(len(test_aucs))]]
ax.errorbar(x + width/2, test_aucs, yerr=errors, fmt='none',
            ecolor='black', capsize=5, capthick=2)

ax.set_xlabel('Model', fontsize=13, fontweight='bold')
ax.set_ylabel('AUC Score', fontsize=13, fontweight='bold')
ax.set_title('Model Performance Comparison', fontsize=15, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=11)
ax.legend(fontsize=11, loc='lower right')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim([0.55, 0.75])

# Add value labels
for bar, auc in zip(bars1, val_aucs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.003,
            f'{auc:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

for bar, auc in zip(bars2, test_aucs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.015,
            f'{auc:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('fig2_model_performance.png', dpi=300, bbox_inches='tight')
print("✓ Saved: fig2_model_performance.png")
plt.show()

# Print summary
print("\n" + "=" * 80)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 80)
for model_name in results:
    auc = results[model_name]['test_auc']
    ci = results[model_name]['test_ci']
    params = results[model_name]['params']
    print(f"\n{model_name}:")
    print(f"  Best hyperparameters: {params}")
    print(f"  Validation AUC: {results[model_name]['val_auc']:.4f}")
    print(f"  Test AUC: {auc:.4f} (95% CI: [{ci[0]:.4f}, {ci[1]:.4f}])")

# ============================================================================
# 1.3 FEATURE IMPORTANCES
# ============================================================================

print("\n" + "=" * 80)
print("[STEP 5] 1.3 Feature Importances Analysis")
print("=" * 80)

lr_model = results['Logistic Regression']['model']
feature_importance = pd.DataFrame({
    'feature': X_imputed.columns,
    'coefficient': lr_model.coef_[0]
}).sort_values('coefficient')

print("\n📈 Top 10 Features INCREASING Readmission Risk:")
top_positive = feature_importance.tail(10)
for idx, row in top_positive.iterrows():
    print(f"  {row['feature']:35s}: {row['coefficient']:+.4f}")

print("\n📉 Top 10 Features DECREASING Readmission Risk:")
top_negative = feature_importance.head(10)
for idx, row in top_negative.iterrows():
    print(f"  {row['feature']:35s}: {row['coefficient']:+.4f}")

# Visualization
print("\n📊 Creating feature importance plot...")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Positive features
axes[0].barh(range(len(top_positive)), top_positive['coefficient'],
             color='#C1121F', alpha=0.8, edgecolor='black', linewidth=0.5)
axes[0].set_yticks(range(len(top_positive)))
axes[0].set_yticklabels(top_positive['feature'], fontsize=11)
axes[0].set_xlabel('Coefficient Value', fontsize=12, fontweight='bold')
axes[0].set_title('Top 10 Features Increasing Readmission Risk',
                  fontsize=13, fontweight='bold', pad=10)
axes[0].grid(axis='x', alpha=0.3, linestyle='--')
axes[0].axvline(0, color='black', linewidth=1.5)

# Negative features
axes[1].barh(range(len(top_negative)), top_negative['coefficient'],
             color='#0B6623', alpha=0.8, edgecolor='black', linewidth=0.5)
axes[1].set_yticks(range(len(top_negative)))
axes[1].set_yticklabels(top_negative['feature'], fontsize=11)
axes[1].set_xlabel('Coefficient Value', fontsize=12, fontweight='bold')
axes[1].set_title('Top 10 Features Decreasing Readmission Risk',
                  fontsize=13, fontweight='bold', pad=10)
axes[1].grid(axis='x', alpha=0.3, linestyle='--')
axes[1].axvline(0, color='black', linewidth=1.5)

plt.tight_layout()
plt.savefig('fig3_feature_importances.png', dpi=300, bbox_inches='tight')
print("✓ Saved: fig3_feature_importances.png")
plt.show()

# ============================================================================
# 1.4 SUBGROUP EVALUATION
# ============================================================================

print("\n" + "=" * 80)
print("[STEP 6] 1.4 Subgroup Performance Evaluation")
print("=" * 80)

# Select best model
best_model_name = max(results.items(), key=lambda x: x[1]['test_auc'])[0]
best_predictions = results[best_model_name]['test_pred']

print(f"\n✓ Using best model: {best_model_name}")
print(f"  Test AUC: {results[best_model_name]['test_auc']:.4f}")

# Find threshold for ~95% accuracy
print(f"\n🎯 Finding threshold for 95% accuracy...")
thresholds = np.linspace(0, 1, 1000)
accuracies = [accuracy_score(y_test, (best_predictions >= t).astype(int)) for t in thresholds]

target_acc = 0.95
closest_idx = np.argmin(np.abs(np.array(accuracies) - target_acc))
selected_threshold = thresholds[closest_idx]
selected_accuracy = accuracies[closest_idx]

print(f"  Selected threshold: {selected_threshold:.4f}")
print(f"  Achieved accuracy: {selected_accuracy:.4f}")

# Generate binary predictions
y_pred_binary = (best_predictions >= selected_threshold).astype(int)

# Overall metrics
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_binary).ravel()
overall_sens = tp / (tp + fn)
overall_spec = tn / (tn + fp)

print(f"  Overall sensitivity: {overall_sens:.4f}")
print(f"  Overall specificity: {overall_spec:.4f}")

# Create test dataframe
test_df = df_clean.iloc[X_test.index].copy()
test_df['y_true'] = y_test.values
test_df['y_pred'] = y_pred_binary

# Function to calculate metrics with CI
def calc_metrics_with_ci(group_df, n_boot=1000):
    """Calculate accuracy, sensitivity, specificity with 95% CI"""
    y_t = group_df['y_true'].values
    y_p = group_df['y_pred'].values
    
    try:
        tn, fp, fn, tp = confusion_matrix(y_t, y_p).ravel()
    except:
        return None
    
    acc = (tp + tn) / (tp + tn + fp + fn)
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    # Bootstrap for CI
    acc_boot, sens_boot, spec_boot = [], [], []
    for _ in range(n_boot):
        idx = np.random.choice(len(group_df), len(group_df), replace=True)
        boot_df = group_df.iloc[idx]
        yt_b = boot_df['y_true'].values
        yp_b = boot_df['y_pred'].values
        
        try:
            tn_b, fp_b, fn_b, tp_b = confusion_matrix(yt_b, yp_b).ravel()
            acc_boot.append((tp_b + tn_b) / (tp_b + tn_b + fp_b + fn_b))
            sens_boot.append(tp_b / (tp_b + fn_b) if (tp_b + fn_b) > 0 else 0)
            spec_boot.append(tn_b / (tn_b + fp_b) if (tn_b + fp_b) > 0 else 0)
        except:
            continue
    
    return {
        'accuracy': acc,
        'acc_ci': np.percentile(acc_boot, [2.5, 97.5]) if acc_boot else [acc, acc],
        'sensitivity': sens,
        'sens_ci': np.percentile(sens_boot, [2.5, 97.5]) if sens_boot else [sens, sens],
        'specificity': spec,
        'spec_ci': np.percentile(spec_boot, [2.5, 97.5]) if spec_boot else [spec, spec],
        'n': len(group_df)
    }

print("\n📊 Calculating subgroup metrics (this takes ~1 minute)...")

# AGE subgroups
age_metrics = {}
for age_group in sorted(test_df['age'].unique()):
    if pd.notna(age_group):
        group_df = test_df[test_df['age'] == age_group]
        if len(group_df) >= 50:
            metrics = calc_metrics_with_ci(group_df, n_boot=500)
            if metrics:
                age_metrics[age_group] = metrics

# GENDER subgroups
gender_metrics = {}
for gender in ['Male', 'Female']:
    group_df = test_df[test_df['gender'] == gender]
    if len(group_df) > 0:
        metrics = calc_metrics_with_ci(group_df, n_boot=500)
        if metrics:
            gender_metrics[gender] = metrics

# RACE subgroups
race_metrics = {}
for race in test_df['race'].unique():
    if pd.notna(race) and race != '?':
        group_df = test_df[test_df['race'] == race]
        if len(group_df) >= 100:
            metrics = calc_metrics_with_ci(group_df, n_boot=500)
            if metrics:
                race_metrics[race] = metrics

# Print results
print("\n" + "=" * 80)
print("SUBGROUP PERFORMANCE SUMMARY")
print("=" * 80)

print("\nAGE GROUPS:")
for group in sorted(age_metrics.keys()):
    m = age_metrics[group]
    print(f"  {group:20s}: Acc={m['accuracy']:.3f} [{m['acc_ci'][0]:.3f}, {m['acc_ci'][1]:.3f}], " +
          f"Sens={m['sensitivity']:.3f}, Spec={m['specificity']:.3f}, n={m['n']:,}")

print("\nGENDER:")
for group in ['Male', 'Female']:
    if group in gender_metrics:
        m = gender_metrics[group]
        print(f"  {group:20s}: Acc={m['accuracy']:.3f} [{m['acc_ci'][0]:.3f}, {m['acc_ci'][1]:.3f}], " +
              f"Sens={m['sensitivity']:.3f}, Spec={m['specificity']:.3f}, n={m['n']:,}")

print("\nRACE:")
for group in sorted(race_metrics.keys()):
    m = race_metrics[group]
    print(f"  {group:20s}: Acc={m['accuracy']:.3f} [{m['acc_ci'][0]:.3f}, {m['acc_ci'][1]:.3f}], " +
          f"Sens={m['sensitivity']:.3f}, Spec={m['specificity']:.3f}, n={m['n']:,}")

# Visualization
print("\n📊 Creating subgroup performance visualization...")

fig, axes = plt.subplots(3, 3, figsize=(18, 14))

def plot_metric_with_ci(ax, metrics_dict, metric_name, ci_name, ylabel, title):
    """Plot metric with confidence intervals"""
    groups = list(metrics_dict.keys())
    values = [metrics_dict[g][metric_name] for g in groups]
    cis = [metrics_dict[g][ci_name] for g in groups]
    
    x = range(len(groups))
    ax.bar(x, values, alpha=0.75, color='steelblue', edgecolor='black', linewidth=1)
    
    # Error bars
    ci_lower = [values[i] - cis[i][0] for i in range(len(values))]
    ci_upper = [cis[i][1] - values[i] for i in range(len(values))]
    ax.errorbar(x, values, yerr=[ci_lower, ci_upper], fmt='none',
                ecolor='black', capsize=5, capthick=2)
    
    ax.set_xticks(x)
    ax.set_xticklabels(groups, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_ylim([0, 1.05])

# Age plots
plot_metric_with_ci(axes[0, 0], age_metrics, 'accuracy', 'acc_ci',
                    'Accuracy', 'Accuracy by Age Group')
plot_metric_with_ci(axes[0, 1], age_metrics, 'sensitivity', 'sens_ci',
                    'Sensitivity', 'Sensitivity by Age Group')
plot_metric_with_ci(axes[0, 2], age_metrics, 'specificity', 'spec_ci',
                    'Specificity', 'Specificity by Age Group')

# Gender plots
plot_metric_with_ci(axes[1, 0], gender_metrics, 'accuracy', 'acc_ci',
                    'Accuracy', 'Accuracy by Gender')
plot_metric_with_ci(axes[1, 1], gender_metrics, 'sensitivity', 'sens_ci',
                    'Sensitivity', 'Sensitivity by Gender')
plot_metric_with_ci(axes[1, 2], gender_metrics, 'specificity', 'spec_ci',
                    'Specificity', 'Specificity by Gender')

# Race plots
plot_metric_with_ci(axes[2, 0], race_metrics, 'accuracy', 'acc_ci',
                    'Accuracy', 'Accuracy by Race')
plot_metric_with_ci(axes[2, 1], race_metrics, 'sensitivity', 'sens_ci',
                    'Sensitivity', 'Sensitivity by Race')
plot_metric_with_ci(axes[2, 2], race_metrics, 'specificity', 'spec_ci',
                    'Specificity', 'Specificity by Race')

plt.tight_layout()
plt.savefig('fig4_subgroup_performance.png', dpi=300, bbox_inches='tight')
print("✓ Saved: fig4_subgroup_performance.png")
plt.show()

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("✅ ANALYSIS COMPLETE!")
print("=" * 80)

print("\n📁 Generated Files (download from Files panel on left):")
print("  1. fig1_demographics.png")
print("  2. fig2_model_performance.png")
print("  3. fig3_feature_importances.png")
print("  4. fig4_subgroup_performance.png")

print("\n📊 Model Performance Summary:")
for model_name in results:
    auc = results[model_name]['test_auc']
    ci = results[model_name]['test_ci']
    print(f"  {model_name:20s}: AUC = {auc:.4f} (95% CI: [{ci[0]:.4f}, {ci[1]:.4f}])")

print(f"\n🏆 Best Model: {best_model_name}")
print(f"  Test AUC: {results[best_model_name]['test_auc']:.4f}")
print(f"  Threshold: {selected_threshold:.4f}")
print(f"  Accuracy: {selected_accuracy:.4f}")

print("\n" + "=" * 80)
print("✓ Copy these results into your report template!")
print("=" * 80)

# Download all figures
print("\n📥 Downloading all figures...")
from google.colab import files
files.download('fig1_demographics.png')
files.download('fig2_model_performance.png')
files.download('fig3_feature_importances.png')
files.download('fig4_subgroup_performance.png')
print("✓ All figures downloaded!")

In [ ]:
================================================================================
CPH 200C - DIABETES READMISSION RISK STRATIFICATION
================================================================================

📤 Please upload: diabetes_130-us_hospitals_for_years_1999-2008.zip
diabetes+130-us+hospitals+for+years+1999-2008.zip
diabetes+130-us+hospitals+for+years+1999-2008.zip(application/zip) - 3170254 bytes, last modified: 4/24/2026 - 100% done
Saving diabetes+130-us+hospitals+for+years+1999-2008.zip to diabetes+130-us+hospitals+for+years+1999-2008.zip
✓ Dataset extracted!

================================================================================
[STEP 1] Loading Dataset
================================================================================

✓ Loaded 101,766 hospital encounters
✓ Unique patients: 71,518
✓ Features: 50

Target variable distribution:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

✓ Overall 30-day readmission rate: 11.16%

================================================================================
[STEP 2] 1.1 DATA EXPLORATION - Demographics Analysis
================================================================================

📊 Readmission Rates by Age Group:

             rate_%      n
age                       
[0-10)     1.863354    161
[10-20)    5.788712    691
[20-30)   14.242607   1657
[30-40)   11.231788   3775
[40-50)   10.604027   9685
[50-60)    9.666203  17256
[60-70)   11.128408  22483
[70-80)   11.773055  26068
[80-90)   12.083503  17197
[90-100)  11.099177   2793

📊 Readmission Rates by Gender:

           rate_%      n
gender                  
Female  11.245156  54708
Male    11.061524  47055

📊 Readmission Rates by Race:

                    rate_%      n
race                             
Caucasian        11.290556  76099
AfricanAmerican  11.218116  19210
Hispanic         10.407462   2037
Asian            10.140406    641
Other             9.628154   1506

========================================
KEY FINDINGS:
========================================
• Highest age group readmission: [20-30) (14.2%)
• Females have higher readmission rate than males (11.2% vs 11.1%)
• Highest race group readmission: Caucasian (11.3%)

📊 Creating demographic visualization...
✓ Saved: fig1_demographics.png


================================================================================
[STEP 3] 1.2 Data Preprocessing
================================================================================

📋 Feature Engineering:
  Total features: 47
  Categorical features: 36
  Numerical features: 11

⚠️  Missing data:
  race: 2.2% missing
  weight: 96.9% missing
  payer_code: 39.6% missing
  medical_specialty: 49.1% missing
  diag_1: 0.0% missing

🔄 Encoding categorical variables...
🔄 Imputing missing values...

📊 Splitting data (60% train, 20% val, 20% test)...
  Train: 61,059 samples (11.2% positive)
  Val:   20,353 samples (11.2% positive)
  Test:  20,354 samples (11.2% positive)
✓ Preprocessing complete!

================================================================================
[STEP 4] 1.2 Model Training
================================================================================

🤖 [1/3] Training Logistic Regression...
  Searching hyperparameters: C ∈ {0.001, 0.01, 0.1, 1, 10, 100}
  ✓ Best params: {'C': 0.001, 'max_iter': 1000, 'penalty': 'l2'}
  ✓ Validation AUC: 0.6401
  ✓ Test AUC: 0.6449 (95% CI: [0.6320, 0.6569])

🌲 [2/3] Training Random Forest...
  Searching hyperparameters (this takes ~2 minutes)...
  ✓ Best params: {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 200}
  ✓ Validation AUC: 0.6607
  ✓ Test AUC: 0.6685 (95% CI: [0.6574, 0.6809])

🧠 [3/3] Training Neural Network...
  Searching hyperparameters...
  ✓ Best params: {'alpha': 0.001, 'hidden_layer_sizes': (50,), 'learning_rate_init': 0.01}
  ✓ Validation AUC: 0.6235
  ✓ Test AUC: 0.6237 (95% CI: [0.6116, 0.6357])

📊 Creating model comparison plot...
✓ Saved: fig2_model_performance.png


================================================================================
MODEL PERFORMANCE SUMMARY
================================================================================

Logistic Regression:
  Best hyperparameters: {'C': 0.001, 'max_iter': 1000, 'penalty': 'l2'}
  Validation AUC: 0.6401
  Test AUC: 0.6449 (95% CI: [0.6320, 0.6569])

Random Forest:
  Best hyperparameters: {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 200}
  Validation AUC: 0.6607
  Test AUC: 0.6685 (95% CI: [0.6574, 0.6809])

Neural Network:
  Best hyperparameters: {'alpha': 0.001, 'hidden_layer_sizes': (50,), 'learning_rate_init': 0.01}
  Validation AUC: 0.6235
  Test AUC: 0.6237 (95% CI: [0.6116, 0.6357])

================================================================================
[STEP 5] 1.3 Feature Importances Analysis
================================================================================

📈 Top 10 Features INCREASING Readmission Risk:
  payer_code                         : +0.0148
  gender                             : +0.0163
  age                                : +0.0298
  number_emergency                   : +0.0366
  time_in_hospital                   : +0.0370
  num_medications                    : +0.0402
  number_diagnoses                   : +0.0726
  diabetesMed                        : +0.0810
  discharge_disposition_id           : +0.1290
  number_inpatient                   : +0.2986

📉 Top 10 Features DECREASING Readmission Risk:
  metformin                          : -0.0574
  num_procedures                     : -0.0337
  chlorpropamide                     : -0.0293
  pioglitazone                       : -0.0252
  insulin                            : -0.0248
  glimepiride                        : -0.0228
  acarbose                           : -0.0206
  tolbutamide                        : -0.0186
  admission_type_id                  : -0.0183
  diag_1                             : -0.0166

📊 Creating feature importance plot...
✓ Saved: fig3_feature_importances.png


================================================================================
[STEP 6] 1.4 Subgroup Performance Evaluation
================================================================================

✓ Using best model: Random Forest
  Test AUC: 0.6685

🎯 Finding threshold for 95% accuracy...
  Selected threshold: 0.3103
  Achieved accuracy: 0.8893
  Overall sensitivity: 0.0216
  Overall specificity: 0.9983

📊 Calculating subgroup metrics (this takes ~1 minute)...

================================================================================
SUBGROUP PERFORMANCE SUMMARY
================================================================================

AGE GROUPS:
  [10-20)             : Acc=0.954 [0.915, 0.985], Sens=0.000, Spec=1.000, n=130
  [20-30)             : Acc=0.880 [0.847, 0.914], Sens=0.279, Spec=0.972, n=324
  [30-40)             : Acc=0.888 [0.866, 0.911], Sens=0.013, Spec=0.994, n=725
  [40-50)             : Acc=0.902 [0.889, 0.915], Sens=0.064, Spec=0.993, n=1,913
  [50-60)             : Acc=0.916 [0.906, 0.923], Sens=0.040, Spec=0.999, n=3,457
  [60-70)             : Acc=0.885 [0.875, 0.894], Sens=0.011, Spec=1.000, n=4,547
  [70-80)             : Acc=0.881 [0.872, 0.889], Sens=0.008, Spec=1.000, n=5,234
  [80-90)             : Acc=0.876 [0.865, 0.885], Sens=0.002, Spec=1.000, n=3,414
  [90-100)            : Acc=0.872 [0.843, 0.898], Sens=0.000, Spec=1.000, n=576

GENDER:
  Male                : Acc=0.893 [0.887, 0.899], Sens=0.020, Spec=0.999, n=9,430
  Female              : Acc=0.886 [0.880, 0.892], Sens=0.023, Spec=0.998, n=10,924

RACE:
  AfricanAmerican     : Acc=0.890 [0.881, 0.901], Sens=0.030, Spec=0.999, n=3,866
  Asian               : Acc=0.919 [0.870, 0.964], Sens=0.000, Spec=1.000, n=123
  Caucasian           : Acc=0.888 [0.882, 0.893], Sens=0.020, Spec=0.998, n=15,223
  Hispanic            : Acc=0.881 [0.847, 0.911], Sens=0.040, Spec=1.000, n=404
  Other               : Acc=0.928 [0.895, 0.957], Sens=0.000, Spec=1.000, n=276

📊 Creating subgroup performance visualization...
✓ Saved: fig4_subgroup_performance.png


================================================================================
✅ ANALYSIS COMPLETE!
================================================================================

📁 Generated Files (download from Files panel on left):
  1. fig1_demographics.png
  2. fig2_model_performance.png
  3. fig3_feature_importances.png
  4. fig4_subgroup_performance.png

📊 Model Performance Summary:
  Logistic Regression : AUC = 0.6449 (95% CI: [0.6320, 0.6569])
  Random Forest       : AUC = 0.6685 (95% CI: [0.6574, 0.6809])
  Neural Network      : AUC = 0.6237 (95% CI: [0.6116, 0.6357])

🏆 Best Model: Random Forest
  Test AUC: 0.6685
  Threshold: 0.3103
  Accuracy: 0.8893

================================================================================
✓ Copy these results into your report template!
================================================================================

📥 Downloading all figures...
✓ All figures downloaded!